# Elyza-7B Version5 ファインチューニング

このノートブックは、手動でファイルをアップロードして学習を実行する形式です。

## 必要なファイル
1. **train_data_v5.jsonl** (データセット)
2. **config_v5.yaml** (設定ファイル)
3. **train_v5.py** (学習スクリプト)

## 1. 環境セットアップ (Unslothインストール)
Unslothライブラリをインストールし、GPU環境をセットアップします。

In [ ]:
import torch

# GPUチェック
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU検出: {gpu_name}")
    if "T4" not in gpu_name and "A100" not in gpu_name and "L4" not in gpu_name:
        print("⚠️ 注意: T4, L4, A100以外のGPUではUnslothが動作しない可能性があります。")
else:
    raise RuntimeError("❌ GPUが検出されません。「ランタイムのタイプ変更」からGPU (T4) を有効にしてください。")

# Unslothのインストール
print("⏳ Unslothをインストール中... (数分かかります)")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

print("✅ インストール完了")

## 2. Google Driveのマウント
学習結果のバックアップ先としてドライブをマウントします。

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 保存先の作成
drive_save_path = "/content/drive/MyDrive/Elyza_Unsloth_Models"
os.makedirs(drive_save_path, exist_ok=True)
print(f"📂 保存先を設定しました: {drive_save_path}")

## 3. ファイルのアップロード
以下の3つのファイルをアップロードしてください。
- `train_data_v5.jsonl` (または対応するデータセット)
- `config_v5.yaml`
- `train_v5.py`

In [ ]:
from google.colab import files
import os

print("ファイルのアップロードを開始します...\n")
uploaded = files.upload()

# アップロード確認
required_files = ["config_v5.yaml", "train_v5.py"]
for file in required_files:
    if file in uploaded:
        print(f"✅ {file} アップロード完了")
    else:
        print(f"⚠️ {file} が見つかりません。ファイル名を確認してください。")

# データセット確認 (拡張子 .jsonl を探す)
dataset_exists = any(f.endswith(".jsonl") for f in uploaded.keys())
if dataset_exists:
    print("✅ データセット(.jsonl) アップロード完了")
else:
    print("⚠️ .jsonlファイルが見つかりません。")

## 4. 学習の実行
アップロードしたスクリプトと設定ファイルを使用して、Unslothによる高速学習を開始します。

In [ ]:
# 設定ファイルのパスを指定して学習実行
!python train_v5.py --config_path config_v5.yaml

## 5. 推論テスト (動作確認)
学習直後のモデルをロードして、簡単な推論テストを行います。

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- 設定 ---
adapter_path = "./results_unsloth/final_adapter" # 学習結果の出力先
max_seq_length = 4096
# ------------

if os.path.exists(adapter_path):
    print("⏳ モデルをロード中...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = adapter_path,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    # テスト用プロンプト
    instruction = "IT法務コンサルタントとして、以下の利用規約案のリスクを指摘してください。"
    input_text = "「当社は、ユーザーが投稿したコンテンツを、理由の如何を問わず、ユーザーへの通知なく削除できるものとします。」"

    system_message = (
        "あなたはIT法務およびコンプライアンスの専門コンサルタントです。\n"
        "以下のフォーマットに厳密に従って回答してください。\n\n"
        "リスクレベル：（低・中・高のいずれかを明記）\n"
        "該当法：（関連する法律名と条文番号）\n"
        "理由：（法的根拠と違反の可能性について詳細に解説）\n"
        "修正案：（具体的な改善提案。必ず記述すること）"
    )

    prompt = f"""<s>[INST] <<SYS>>\n{system_message}\n<</SYS>>\n\n{instruction}\n\n入力:\n{input_text} [/INST]"""
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    print("🤖 生成中...")
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 1024, 
        use_cache = True,
        temperature = 0.6,
        top_p = 0.9
    )

    print("\n" + "="*30 + " 結果 " + "="*30)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("[/INST]")[-1].strip())
else:
    print("❌ 学習済みモデルが見つかりません。Step 4が完了しているか確認してください。")

## 6. Google Driveへ保存
学習済みアダプターをGoogle Driveへコピーします。

In [ ]:
import shutil
import os
import datetime

source_dir = "./results_unsloth/final_adapter"
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dest_dir = f"/content/drive/MyDrive/Elyza_Unsloth_Models/Elyza-7B-v5_{timestamp}"

if os.path.exists(source_dir):
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(source_dir, dest_dir)
    print(f"✅ Google Driveに保存しました: {dest_dir}")
else:
    print("❌ 保存元ディレクトリが見つかりません。")